In [ ]:
# TemporalBench v1 — Baseline Tier
# Difficulty: Well-separated validity windows, no noise injection
# Task families: AsOfQA, ChangeDetection, CausalQuery
# !pip install kaggle-environments

import json
import os
import random
import numpy as np
from typing import Dict, List, Any, Optional

In [ ]:
# ---------------------------------------------------------------------------

In [ ]:
# KAGGLE BENCHMARK REGISTRATION (required for all task notebooks)

In [ ]:
# ---------------------------------------------------------------------------

# @kbench.task decorator registers this notebook as a Kaggle benchmark task.
# The decorator accepts:
#   name        : Display name for the leaderboard
#   description : What the task measures
#   tasks       : Number of evaluated conditions (not individual questions)

# @kbench.task(name="TemporalBench v1", description="AsOfQA, ChangeDetection, CausalQuery — baseline conditions", tasks=3)

In [ ]:
# ---------------------------------------------------------------------------

In [ ]:
# TEMPORAL BENCHMARK CORE

In [ ]:
# ---------------------------------------------------------------------------

In [ ]:
class Fact:

In [ ]:
def __init__(self, key: str, value: str, valid_from: int, valid_to: int, accesses: int = 0):
        self.key = key
        self.value = value
        self.valid_from = valid_from
        self.valid_to = valid_to  # exclusive upper bound
        self.accesses = accesses

In [ ]:
def is_valid_at(self, day: int) -> bool:
        return self.valid_from <= day < self.valid_to

In [ ]:
def age(self, day: int) -> float:
        return day - self.valid_from

In [ ]:
def decay_score(self, day: int, half_life: float = 50.0) -> float:
        return np.exp(-0.693 * self.age(day) / half_life)

In [ ]:
def attention_score(self) -> float:
        return 1.0 + 0.001 * np.log1p(self.accesses)

In [ ]:
class TemporalAttentionStore:
    """Standard decay + attention baseline (System D in the paper)."""

In [ ]:
def __init__(self, half_life: float = 50.0, temporal_weight: float = 0.9, attention_weight: float = 0.1):
        self.half_life = half_life
        self.temporal_weight = temporal_weight
        self.attention_weight = attention_weight
        self.facts: Dict[str, List[Fact]] = {}

In [ ]:
def put(self, key: str, value: str, valid_from: int, valid_to: int, accesses: int = 0):
        if key not in self.facts:
            self.facts[key] = []
        self.facts[key].append(Fact(key, value, valid_from, valid_to, accesses))

In [ ]:
def get(self, key: str, as_of_day: Optional[int] = None) -> Optional[str]:
        if key not in self.facts:
            return None
        
        candidates = [f for f in self.facts[key] if f.is_valid_at(as_of_day)]
        if not candidates:
            return None
        
        # Score: decay * attention (System D — the decay baseline)
        scores = []
        for f in candidates:
            ds = f.decay_score(as_of_day, self.half_life)
            as_ = f.attention_score()
            score = self.temporal_weight * ds + self.attention_weight * as_
            scores.append((score, f))
        
        best = max(scores, key=lambda x: x[0])
        return best[1].value

In [ ]:
class TemporalAttentionStoreWithValidity:
    """D_revised — validity windows + decay + attention (System D_revised in the paper)."""

In [ ]:
def __init__(self, half_life: float = 50.0):
        self.half_life = half_life
        self.facts: Dict[str, List[Fact]] = {}

In [ ]:
def put(self, key: str, value: str, valid_from: int, valid_to: int, accesses: int = 0):
        if key not in self.facts:
            self.facts[key] = []
        self.facts[key].append(Fact(key, value, valid_from, valid_to, accesses))

In [ ]:
def get(self, key: str, as_of_day: Optional[int] = None) -> Optional[str]:
        """Hard validity gate — only valid facts compete."""
        if key not in self.facts:
            return None
        
        # HARD FILTER: only facts valid at query_day
        candidates = [f for f in self.facts[key] if f.is_valid_at(as_of_day)]
        if not candidates:
            return None
        
        # Within valid set: rank by decay * attention
        scores = []
        for f in candidates:
            ds = f.decay_score(as_of_day, self.half_life)
            as_ = f.attention_score()
            score = ds * as_
            scores.append((score, f))
        
        best = max(scores, key=lambda x: x[0])
        return best[1].value

In [ ]:
def load_benchmark(path: str) -> tuple:
    """Load question/fact/event JSONL files for a benchmark tier."""
    with open(path + '_questions.jsonl') as f:
        questions = [json.loads(line) for line in f]
    with open(path + '_facts.jsonl') as f:
        facts_data = [json.loads(line) for line in f]
    with open(path + '_events.jsonl') as f:
        events_data = [json.loads(line) for line in f]
    return questions, facts_data, events_data

In [ ]:
def build_store(facts_data: List[Dict], events_data: List[Dict], store_class) -> tuple:
    """Populate a TemporalAttentionStore from fact + event JSONL.
    
    Returns (store, event_log) where event_log is sorted by day.
    """
    store = store_class()
    
    # Add facts
    for fact in facts_data:
        store.put(
            key=fact['key'],
            value=fact['value'],
            valid_from=fact['valid_from'],
            valid_to=fact['valid_to'],
            accesses=fact.get('accesses', 0)
        )
    
    # Build event log sorted by day
    event_log = sorted(events_data, key=lambda x: x['day'])
    
    return store, event_log

In [ ]:
def replay_events(store, events_data: List[Dict], up_to_day: int):
    """Replay events up to (and including) up_to_day to populate store access counts."""
    for ev in events_data:
        if ev['day'] > up_to_day:
            break
        # Event types: 'fact_access' increments accesses for a fact
        if ev.get('type') == 'fact_access':
            for key_facts in store.facts.values():
                for f in key_facts:
                    if f.valid_from == ev.get('valid_from') and f.valid_to == ev.get('valid_to'):
                        f.accesses = ev.get('accesses', f.accesses)

In [ ]:
def score_task(question: Dict, predicted: Optional[str], store, event_log, store_class) -> float:
    """Score a single question. Returns 1.0 for correct, 0.0 for incorrect."""
    task_family = question.get('task_family', 'AsOfQA')
    as_of_day = question.get('as_of_day')
    
    if task_family == 'AsOfQA':
        # Query: "As of day X, what was true about subject Y in domain Z?"
        # Expected: the value of subject Y in domain Z at day X
        domain = question.get('domain')
        subject = question.get('subject')
        key = f"{domain}:{subject}"
        answer = question.get('answer')
        
        if answer:
            return 1.0 if predicted == answer else 0.0
        return 1.0 if predicted else 0.0
    
    elif task_family == 'ChangeDetection':
        # Expected: day when status changed from X to Y
        expected = question.get('expected_day') or question.get('answer')
        return 1.0 if str(predicted).strip() == str(expected).strip() else 0.0
    
    elif task_family == 'CausalQuery':
        # Expected: causal chain explanation (partial credit handled by evaluator)
        expected = question.get('expected_answer') or question.get('answer')
        if not expected or not predicted:
            return 0.0
        # Simple token overlap as proxy
        pred_tokens = set(str(predicted).lower().split())
        exp_tokens = set(str(expected).lower().split())
        overlap = len(pred_tokens & exp_tokens)
        return overlap / max(len(exp_tokens), 1)
    
    return 0.0

In [ ]:
def run_evaluation(questions: List[Dict], facts_data: List[Dict], events_data: List[Dict],
                   store_class=TemporalAttentionStoreWithValidity) -> Dict:
    """
    Run full evaluation of a store on all questions.
    Returns per-task-family metrics and overall TRS.
    """
    store, event_log = build_store(facts_data, events_data, store_class)
    
    results = {}
    for q in questions:
        tf = q.get('task_family', 'Unknown')
        if tf not in results:
            results[tf] = {'correct': 0, 'total': 0}
        
        as_of_day = q.get('as_of_day') or q.get('day')
        replay_events(store, events_data, as_of_day)
        
        key = f"{q.get('domain', '')}:{q.get('subject', '')}"
        predicted = store.get(key, as_of_day)
        
        score = score_task(q, predicted, store, event_log, store_class)
        results[tf]['correct'] += score
        results[tf]['total'] += 1
    
    # Compute per-family accuracy
    metrics = {}
    for tf, vals in results.items():
        metrics[tf] = vals['correct'] / vals['total'] if vals['total'] > 0 else 0.0
    
    # TRS = mean accuracy across task families (multiplicative validity handled separately)
    trs = np.mean(list(metrics.values())) if metrics else 0.0
    
    return {'metrics': metrics, 'trs': trs, 'details': results}

In [ ]:
# ---------------------------------------------------------------------------
# MAIN EVALUATION

In [ ]:
# ---------------------------------------------------------------------------

DATA_PATH = '/kaggle/input/datasets/zacharymaronek/temporalbenchmark/temporalbench/data/benchmarks/temporalbench_v1'

print('Loading TemporalBench v1 (baseline tier)...')
questions, facts_data, events_data = load_benchmark(DATA_PATH)
print(f'Loaded {len(questions)} questions, {len(facts_data)} facts, {len(events_data)} events')

print('\nEvaluating D_revised (Validity Windows)...')
results_revised = run_evaluation(questions, facts_data, events_data, TemporalAttentionStoreWithValidity)
print(f'TRS: {results_revised["trs"]:.4f}')
for tf, acc in results_revised['metrics'].items():
    print(f'  {tf}: {acc:.1%}')

print('\nEvaluating D (Decay baseline)...')
results_d = run_evaluation(questions, facts_data, events_data, TemporalAttentionStore)
print(f'TRS: {results_d["trs"]:.4f}')
for tf, acc in results_d['metrics'].items():
    print(f'  {tf}: {acc:.1%}')

print('\n--- SUMMARY ---')
print(f'D_revised (Validity Windows): {results_revised["trs"]:.1%}')
print(f'D (Decay baseline):           {results_d["trs"]:.1%}')
print(f'Improvement:                  {(results_revised["trs"] - results_d["trs"]):.1%}')